# Advanced Analytics

CUSUM charts, Bayesian forecasting, clustering, and statistical analyses.

In [ ]:
import sys
sys.path.insert(0, '/home/user/nyc_data')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.cluster import KMeans
from scipy import stats

print('✓ Environment initialized')

## CUSUM Control Chart

In [ ]:
# Generate time series data
np.random.seed(42)
n = 100
x = np.arange(n)
violations_per_day = np.random.poisson(2.5, n) + np.where(x > 70, 5, 0)

# Calculate CUSUM
target = np.mean(violations_per_day[:70])
cusum = np.zeros(n)
cusum_neg = np.zeros(n)

for i in range(1, n):
    cusum[i] = max(0, cusum[i-1] + violations_per_day[i] - target)
    cusum_neg[i] = min(0, cusum_neg[i-1] + violations_per_day[i] - target)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x, y=violations_per_day,
    mode='lines', name='Actual',
    line=dict(color='gray')
))

fig.add_trace(go.Scatter(
    x=x, y=cusum,
    mode='lines', name='CUSUM+',
    line=dict(color='red')
))

fig.update_layout(
    title='CUSUM Control Chart — Violation Detection',
    xaxis_title='Day',
    yaxis_title='CUSUM Value',
    height=400
)

fig.show()

## Bayesian Confidence Intervals

In [ ]:
# Simulate Bayesian posteriors for completion rates
boroughs = ['MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS', 'STATEN_ISLAND']
completions = [75, 62, 58, 68, 45]
totals = [100, 100, 100, 100, 100]

alpha = 1
beta = 1

x_vals = np.linspace(0, 1, 100)

fig = go.Figure()

for i, (borough, comp, total) in enumerate(zip(boroughs, completions, totals)):
    alpha_post = alpha + comp
    beta_post = beta + (total - comp)
    
    posterior = stats.beta.pdf(x_vals, alpha_post, beta_post)
    
    fig.add_trace(go.Scatter(
        x=x_vals, y=posterior,
        mode='lines',
        name=borough,
        fill='tozeroy'
    ))

fig.update_layout(
    title='Bayesian Posteriors — Ramp Completion Rates',
    xaxis_title='Completion Rate',
    yaxis_title='Posterior Density',
    height=400
)

fig.show()

## K-Means Clustering

In [ ]:
# Generate spatial data for clustering
np.random.seed(42)
n_points = 200
cluster_centers = np.array([[0, 0], [5, 5], [-5, 5]])

X = np.vstack([
    cluster_centers[0] + np.random.randn(n_points//3, 2) * 0.5,
    cluster_centers[1] + np.random.randn(n_points//3, 2) * 0.5,
    cluster_centers[2] + np.random.randn(n_points - 2*(n_points//3), 2) * 0.5
])

# Fit K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X)

fig = px.scatter(
    x=X[:, 0], y=X[:, 1],
    color=labels.astype(str),
    title='K-Means Clustering of Inspection Locations',
    labels={'color': 'Cluster'},
    height=400
)

fig.add_scatter(
    x=kmeans.cluster_centers_[:, 0],
    y=kmeans.cluster_centers_[:, 1],
    mode='markers',
    name='Centroids',
    marker=dict(size=15, color='red', symbol='x')
)

fig.show()

## Correlation Analysis

In [ ]:
# Generate synthetic metrics data
n = 50
metrics_data = pd.DataFrame({
    'Inspections': np.random.poisson(20, n),
    'Violations': np.random.poisson(50, n),
    'Completion_Rate': np.random.uniform(0.4, 0.9, n),
    'Budget_Spent': np.random.uniform(50000, 500000, n),
    'Days_Overdue': np.random.poisson(5, n)
})

# Calculate correlation matrix
corr_matrix = metrics_data.corr()

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    title='Correlation Matrix — Key Metrics',
    height=400,
    text_auto='.2f'
)

fig.show()